# EDA: Анализ данных с обоснованием решений pipeline

**Changellenge Cup IT 2026 | Кейс МТС**

Этот notebook содержит анализ исходных данных, который напрямую обосновывает каждое ключевое решение в pipeline. Для каждого наблюдения указано: что нашли, какое решение приняли, почему.

**Структура:**
1. Обзор данных и пропуски: обоснование приоритета Источника Б
2. Геометрия и аномалии: обоснование очистки
3. Высоты и этажность: обоснование target leak и log target
4. Теги и назначение: обоснование feature engineering
5. Пространственный анализ: обоснование matching подхода
6. Итоговые решения

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from shapely import wkt

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

df_a = pd.read_csv('cup_it_example_src_A.csv', index_col=0)
df_b = pd.read_csv('cup_it_example_src_B.csv', index_col=0, low_memory=False)
print(f'Источник А: {df_a.shape}')
print(f'Источник Б: {df_b.shape}')

---
## 1. Покрытие данных: почему Б — приоритетный источник

**Вопрос:** Какому источнику доверять? Откуда брать высоту, геометрию, атрибуты?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Покрытие А
cols_a = {'geometry': df_a['geometry'].notna().mean(),
          'tags': df_a['tags'].notna().mean(),
          'area_sq_m': df_a['area_sq_m'].notna().mean(),
          'gkh_address': df_a['gkh_address'].notna().mean(),
          'gkh_floor_max': df_a['gkh_floor_count_max'].notna().mean()}
axes[0].barh(list(cols_a.keys()), [v*100 for v in cols_a.values()], color='#1f77b4')
axes[0].set_xlabel('Заполненность, %')
axes[0].set_title(f'Источник А ({len(df_a):,} записей)')
axes[0].set_xlim(0, 105)
for i, (k, v) in enumerate(cols_a.items()):
    axes[0].text(v*100+1, i, f'{v*100:.1f}%', va='center', fontsize=10)

# Покрытие Б
cols_b = {'wkt': df_b['wkt'].notna().mean(),
          'height': (df_b['height'].notna() & (df_b['height'] > 0)).mean(),
          'stairs': df_b['stairs'].notna().mean(),
          'purpose': df_b['purpose_of_building'].notna().mean(),
          'name_street': df_b['name_street'].notna().mean(),
          'district': df_b['district'].notna().mean()}
axes[1].barh(list(cols_b.keys()), [v*100 for v in cols_b.values()], color='#d62728')
axes[1].set_xlabel('Заполненность, %')
axes[1].set_title(f'Источник Б ({len(df_b):,} записей)')
axes[1].set_xlim(0, 105)
for i, (k, v) in enumerate(cols_b.items()):
    axes[1].text(v*100+1, i, f'{v*100:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('\n=== ВЫВОД ===')
print('Б имеет высоту для 99.96% зданий. А — 0%.')
print('А имеет ЖКХ этажность для ~10%. Это недостаточно для самостоятельного определения высоты.')
print('→ РЕШЕНИЕ: Б — приоритетный источник для высоты и геометрии.')
print('  А — источник тегов и ЖКХ данных (дополнительные атрибуты).')

---
## 2. Аномалии и очистка: что выкидываем и почему

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 2.1 Площадь А — мелкие объекты
area_a = df_a['area_sq_m']
axes[0,0].hist(np.log10(area_a.clip(lower=0.1)), bins=60, edgecolor='white', color='#1f77b4')
for t, c, l in [(8, 'red', '8 м² (наш порог)'), (20, 'orange', '20 м² (OSM стандарт)')]:
    axes[0,0].axvline(np.log10(t), color=c, ls='--', lw=2, label=l)
axes[0,0].legend(fontsize=9)
axes[0,0].set_xlabel('log10(площадь, м²)')
axes[0,0].set_title(f'А: распределение площади')

# Примеры мелких
small = df_a[df_a['area_sq_m'] < 10]
small_tags = small['tags'].astype(str).str.strip('[]').str.replace("'", '').str.split(',').str[0].str.strip()
axes[0,1].barh(small_tags.value_counts().head(8).index, small_tags.value_counts().head(8).values, color='#ff7f0e')
axes[0,1].set_xlabel('Количество')
axes[0,1].set_title(f'А: теги зданий < 10 м² ({len(small):,} шт)')

# 2.2 Высота Б — аномалии
h = df_b['height'].dropna()
axes[1,0].hist(h.clip(upper=100), bins=60, edgecolor='white', color='#d62728')
axes[1,0].set_xlabel('Высота, м')
axes[1,0].set_title(f'Б: распределение высоты (медиана={h.median():.1f} м)')

# Аномально высокие
very_tall = df_b[df_b['height'] > 100]
axes[1,1].barh(range(len(very_tall)), very_tall['height'].values, color='#d62728')
axes[1,1].set_xlabel('Высота, м')
axes[1,1].set_title(f'Б: здания > 100 м ({len(very_tall)} шт)')

plt.tight_layout()
plt.show()

print('\n=== ВЫВОДЫ ===')
print(f'1. А: {(area_a < 8).sum():,} зданий < 8 м² — "жилое здание" 2 м² физически невозможно.')
print(f'   Индустрия (OSM pipeline) ставит порог 20-40 м². Наш 8 м² — консервативнее.')
print(f'   → РЕШЕНИЕ: фильтр < 8 м²')
print(f'2. Б: {len(very_tall)} зданий > 100 м. Максимум {h.max():.0f} м = Лахта Центр (реальна).')
print(f'   → РЕШЕНИЕ: фильтр > 500 м (сохраняем Лахту, убираем ошибки)')
print(f'3. А: 281 MULTIPOLYGON → explode в отдельные записи')
print(f'4. Б: 66 пропущенных высот → восстановить через stairs × avg_floor_height')

---
## 3. Высота vs этажность: target leak и log target

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3.1 stairs vs height — прямая зависимость
valid = df_b.dropna(subset=['stairs', 'height'])
valid = valid[(valid['stairs'] > 0) & (valid['height'] > 0)]
sample = valid.sample(min(3000, len(valid)), random_state=42)
axes[0].scatter(sample['stairs'], sample['height'], s=3, alpha=0.2, edgecolors='none')
axes[0].plot([0, 30], [0, 90], 'r--', lw=1, label='height = stairs × 3')
axes[0].set_xlabel('Этажность (stairs)')
axes[0].set_ylabel('Высота, м')
axes[0].set_title('stairs vs height — почти линейная зависимость')
axes[0].legend()

# 3.2 Консистентность stairs * avg ≈ height
valid['calc_h'] = valid['stairs'] * valid['avg_floor_height']
valid['diff'] = (valid['height'] - valid['calc_h']).abs()
pct_ok = (valid['diff'] < 1).mean() * 100
axes[1].hist(valid['diff'].clip(upper=20), bins=50, edgecolor='white', color='#2ca02c')
axes[1].axvline(1, color='red', ls='--', label=f'< 1м: {pct_ok:.1f}%')
axes[1].set_xlabel('|height - stairs×avg|, м')
axes[1].set_title('Консистентность: stairs × avg ≈ height')
axes[1].legend()

# 3.3 Распределение высоты — right skew → log target
axes[2].hist(h[h > 0], bins=60, edgecolor='white', color='#d62728', alpha=0.5, label='height')
ax2 = axes[2].twinx()
ax2.hist(np.log1p(h[h > 0]), bins=60, edgecolor='white', color='#1f77b4', alpha=0.5, label='log1p(height)')
axes[2].set_xlabel('Высота')
axes[2].set_title('Right skew → log target лучше для мелких')
axes[2].legend(loc='upper right')
ax2.legend(loc='center right')

plt.tight_layout()
plt.show()

print('\n=== ВЫВОДЫ ===')
print(f'1. stairs ≈ height/3 — это TARGET LEAK. Если дать модели stairs,')
print(f'   она "запомнит" его вместо обучения на геометрических паттернах.')
print(f'   stairs = 100% NaN для only_A (тех для кого предсказываем).')
print(f'   → РЕШЕНИЕ: убрать stairs, gkh_floor_max/min из фичей модели')
print(f'2. Распределение высоты right-skewed (медиана 6.6, mean 14.2).')
print(f'   Без log модель оптимизирует MAE на высоких зданиях, завышает для мелких.')
print(f'   → РЕШЕНИЕ: log1p(height) как target')

---
## 4. Теги и назначение: что знаем о зданиях без высоты

In [ ]:
# Парсим теги А
df_a['tag_main'] = df_a['tags'].astype(str).str.strip('[]').str.replace("'", '').str.split(',').str[0].str.strip()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Теги А
tag_counts = df_a['tag_main'].value_counts().head(12)
tag_counts.plot.barh(ax=axes[0], color='#1f77b4')
axes[0].set_xlabel('Количество зданий')
axes[0].set_title('Источник А: теги')
axes[0].invert_yaxis()

# Назначение Б
purp_counts = df_b['purpose_of_building'].value_counts().head(12)
purp_counts.plot.barh(ax=axes[1], color='#d62728')
axes[1].set_xlabel('Количество зданий')
axes[1].set_title('Источник Б: назначение')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Высота по тегам
merged = df_b.copy()
merged['h'] = merged['height']
print('\nМедиана высоты по назначению (Б):')
for p in purp_counts.head(6).index:
    sub = merged[merged['purpose_of_building'] == p]
    print(f'  {p:<30} median={sub["h"].median():>5.1f} м  n={len(sub):>6,}')

print('\n=== ВЫВОДЫ ===')
print('1. "постройка" — самый частый тег А (48K), но неинформативный:')
print('   под ним гаражи 3м, склады 10м, трансформаторные 5м.')
print('   → РЕШЕНИЕ: разбить по площади (tag_постройка_small/large)')  
print('2. purpose из Б — 46% feature importance, но 100% NaN для only_A.')
print('   Модель переобучается на purpose, плохо предсказывает без него.')
print('   → РЕШЕНИЕ: убрать purpose из модели')
print('3. Target encoding по tag_main: средняя высота по типу здания.')
print('   "жилое здание" → 23.7м, "постройка" → 5.8м.')
print('   → РЕШЕНИЕ: добавить tag_h_mean/median/std как фичи')

---
## 5. Пространственный анализ: обоснование matching подхода

In [ ]:
# Bounding boxes
sample_a = df_a['geometry'].dropna().sample(min(5000, len(df_a)), random_state=42)
sample_b = df_b['wkt'].dropna().sample(min(5000, len(df_b)), random_state=42)

lons_a, lats_a = [], []
for w in sample_a:
    try:
        c = wkt.loads(w).centroid
        lons_a.append(c.x); lats_a.append(c.y)
    except: pass

lons_b, lats_b = [], []
for w in sample_b:
    try:
        c = wkt.loads(w).centroid
        lons_b.append(c.x); lats_b.append(c.y)
    except: pass

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Покрытие
axes[0].scatter(lons_a[:2000], lats_a[:2000], s=8, alpha=0.3, label='А', edgecolors='none')
axes[0].scatter(lons_b[:2000], lats_b[:2000], s=8, alpha=0.3, label='Б', edgecolors='none')
axes[0].set_title('Пространственное покрытие')
axes[0].legend(markerscale=3)
axes[0].set_xlabel('Долгота'); axes[0].set_ylabel('Широта')

# Площади обоих
axes[1].hist(np.log10(df_a['area_sq_m'].clip(lower=0.1)), bins=50, alpha=0.5, label='А', edgecolor='white')
# Площадь Б посчитаем
areas_b = []
for w in sample_b[:3000]:
    try: areas_b.append(wkt.loads(w).area * 111000**2 * 0.5)
    except: pass
axes[1].hist(np.log10(np.clip(areas_b, 0.1, None)), bins=50, alpha=0.5, label='Б', edgecolor='white')
axes[1].set_xlabel('log10(площадь, м²)')
axes[1].set_title('Распределение площадей')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\n=== ВЫВОДЫ ===')
print('1. Оба источника покрывают одну территорию — пространственное сопоставление возможно.')
print('2. Распределения площадей похожи — здания сопоставимы по размеру.')
print('3. А содержит больше мелких объектов (сдвиг влево) — хозяйственные постройки.')
print('   → РЕШЕНИЕ: IoU + overlap ratio + centroid matching')
print('   IoU — для хорошо совпадающих. Overlap — для маленький-внутри-большого.')
print('   Centroid — для сдвинутых (не пересекающихся).')

---
## 6. Адреса: потенциал для валидации

In [ ]:
a_addr = df_a['gkh_address'].notna().sum()
b_addr = df_b['name_street'].notna().sum()

print(f'А с адресом (gkh_address): {a_addr:,} ({a_addr/len(df_a)*100:.1f}%)')
print(f'Б с улицей (name_street):  {b_addr:,} ({b_addr/len(df_b)*100:.1f}%)')
print(f'\nАдресное сопоставление: только ~10% А и ~55% Б имеют адреса.')
print('Как основной метод — недостаточно.')
print('→ РЕШЕНИЕ: использовать адреса для ВАЛИДАЦИИ матчинга (не для матчинга).')
print('  Результат: 87.9% совпадение улиц для matched зданий.')
print('  12.1% несовпадений — угловые здания с двумя адресами.')

---
## 7. Множественные записи в Б: почему не дедупликация

In [ ]:
# Проверяем: несколько записей Б на один адрес — дубли или секции?
b_with_addr = df_b[df_b['name_street'].notna()].copy()
b_with_addr['full_addr'] = b_with_addr['name_street'].astype(str) + ' ' + b_with_addr['number'].astype(str)

addr_counts = b_with_addr['full_addr'].value_counts()
multi = addr_counts[addr_counts > 1]

print(f'Адресов с > 1 записью: {len(multi):,} (из {len(addr_counts):,})')
print(f'Максимум записей на адрес: {multi.max()}')

# Пример: записи на один адрес с РАЗНОЙ высотой
example_addr = multi.index[0]
example = b_with_addr[b_with_addr['full_addr'] == example_addr]
print(f'\nПример: {example_addr}')
print(example[['height', 'stairs', 'purpose_of_building']].to_string())

print('\n=== ВЫВОД ===')
print('Множественные записи — это СЕКЦИИ одного здания с разной высотой.')
print('Это НЕ дубли — каждая секция имеет свою этажность и геометрию.')
print('→ РЕШЕНИЕ: сохранять каждый полигон Б как отдельную запись.')
print('  Не делать unary_union (теряет индивидуальные высоты).')
print('  Лиза (МТС): "нужны самые подробные геометрии — отдельные купола,"')
print('  "отдельные секции с разной высотой."')

---
## 8. ML predicted: что за здания и почему сложно

In [ ]:
# Загрузим результаты pipeline
import geopandas as gpd
final = gpd.read_parquet('./data/final.parquet')
ml = final[final['height_source'] == 'ml_predicted']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Теги ML-predicted
ml['tag_main'].value_counts().head(10).plot.barh(ax=axes[0], color='#ff7f0e')
axes[0].set_title(f'ML predicted: теги ({len(ml):,} зданий)')
axes[0].invert_yaxis()

# Площадь
axes[1].hist(np.log10(ml['area_m2'].clip(lower=0.1)), bins=40, edgecolor='white', color='#ff7f0e')
axes[1].axvline(np.log10(50), color='red', ls='--', label='50 м²')
axes[1].axvline(np.log10(200), color='orange', ls='--', label='200 м²')
axes[1].set_xlabel('log10(площадь, м²)')
axes[1].set_title(f'ML predicted: площадь (median={ml["area_m2"].median():.0f} м²)')
axes[1].legend()

# Предсказанная высота
axes[2].hist(ml['height'].clip(upper=60), bins=40, edgecolor='white', color='#ff7f0e')
axes[2].set_xlabel('Предсказанная высота, м')
axes[2].set_title(f'ML predicted: высота (median={ml["height"].median():.1f} м)')

plt.tight_layout()
plt.show()

# Проблема: OSM cross-check
print('=== ПРОБЛЕМА ===')
print(f'56% ML predicted — "постройка" (median 31 м²). OSM говорит 3 м, мы предсказываем ~14 м.')
print(f'НО 14% — "жилое здание", 5% — крупные (>500 м²): школы, ТЦ, комплексы.')
print(f'Для них ошибка предсказания критична для RF-планирования.')
print(f'\nПричина: spatial lag тянет к высоте соседних жилых домов.')
print(f'"Постройка" рядом с 9-этажкой → модель предсказывает ~14 м вместо 3 м.')
print(f'\n→ РЕШЕНИЕ: убрали purpose (снизило завышение),')
print(f'  добавили size-aware фичи (is_smallest, area_ratio, tag-specific lag),')
print(f'  log target (сжимает диапазон, модель меньше завышает).')
print(f'  OSM ML MAE: 8.86 → 4.68 м (улучшение 47%).')

---
## 9. Сводка решений

| Наблюдение EDA | Решение | Где в pipeline |
|---|---|---|
| Б имеет высоту 99.96%, А — 0% | Приоритет Б для высоты и геометрии | 02_matching |
| 281 MULTIPOLYGON в А | Explode перед матчингом | 01_cleaning |
| 7,649 зданий < 10 м² (ошибки) | Фильтр < 8 м² | 01_cleaning |
| 66 пропущенных высот в Б | Восстановление через stairs × avg | 01_cleaning |
| ~11K за пределами СПб в Б | Фильтр по bbox | 01_cleaning |
| stairs ≈ height/3 | Убрать stairs из модели (target leak) | 03_height_model |
| gkh_floor тоже ≈ height | Убрать gkh_floor из модели (target leak) | 03_height_model |
| purpose = 46% importance, 100% NaN для only_A | Убрать purpose из модели | 03_height_model |
| Распределение высоты right-skewed | log1p(height) как target | 03_height_model |
| "постройка" = 56% ML predicted, median 3 м | Size-aware фичи + tag split | 03_height_model |
| Оба источника покрывают одну территорию | Пространственное сопоставление (IoU) | 02_matching |
| Маленький Б внутри большого А | + overlap ratio | 02_matching |
| Сдвиг полигонов до 20 м | + centroid matching | 02_matching |
| Адреса: 10% А, 55% Б | Адреса для валидации, не для матчинга | 04_validation |
| Множественные записи Б = секции | Одна запись на полигон Б | 02_matching |